In [1]:
from pathlib import Path
import os
print(os.getcwd())
print('\n'.join(str(p) for p in Path('/workspace').iterdir()))

/workspace
/workspace/calphad_grid.py
/workspace/calibration_fit.txt
/workspace/README.md
/workspace/truth_phys.py
/workspace/same-test-rates.md
/workspace/calphad_grid.csv
/workspace/.kernel_llm_logs_1.jsonl
/workspace/.skills
/workspace/kinetics-constants.md
/workspace/calibrate_corrosion.py
/workspace/memory
/workspace/phys_tables_c70600.md
/workspace/phys_tables.md
/workspace/.prompts
/workspace/check_phys.py
/workspace/.config
/workspace/lost+found


In [2]:
import numpy as np, pandas as pd, torch, sys, types, subprocess, json, time
from scipy.optimize import least_squares, differential_evolution, minimize, brentq
OUT=Path('/workspace/review'); OUT.mkdir(exist_ok=True)
torch.set_num_threads(4); torch.set_default_dtype(torch.double)
import truth_phys as TP
BASE=TP.CAL.copy()
df=pd.read_csv('calphad_grid.csv')
print(df.shape, df.sch_conv.value_counts().to_dict())
print('Missing:',df.isna().sum()[df.isna().sum()>0].to_dict())
# Preserve inputs; adapt package layout only in process, not the supplied science.
pkg=types.ModuleType('physics'); pkg.truth_phys=TP; sys.modules['physics']=pkg
import calibrate_corrosion as CC
print('Plan: reproduce the supplied fit and checks; audit grid/rules; refit alternative data/weights/bounds/features; minimize Ni subject to actual mass-loss bars; verify key primary sources.')
for script in ['calibrate_corrosion.py','check_phys.py']:
 p=subprocess.run([sys.executable,script],capture_output=True,text=True)
 (OUT/(script+'.original_run.txt')).write_text(p.stdout+p.stderr)
 print(script,'exit',p.returncode,(p.stdout+p.stderr)[-600:])
print('grid NaNs in truth:',{k:int(v.isnan().sum()) for k,v in TP.grid().V.items()})

(864, 89) {True: 810, False: 54}
Missing: {'primary': 20, 'kou': 84, 'a900_NI': 31, 'a900_AL': 31, 'a900_FE': 31, 'a900_MN': 31, 'a700_NI': 8, 'a700_AL': 8, 'a700_FE': 8, 'a700_MN': 8}
Plan: reproduce the supplied fit and checks; audit grid/rules; refit alternative data/weights/bounds/features; minimize Ni subject to actual mass-loss bars; verify key primary sources.


calibrate_corrosion.py exit 1 Traceback (most recent call last):
  File "/workspace/calibrate_corrosion.py", line 30, in <module>
    from physics import truth_phys as TP  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'physics'



check_phys.py exit 1 Traceback (most recent call last):
  File "/workspace/check_phys.py", line 22, in <module>
    import bofn_alloy as BA  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'bofn_alloy'

grid NaNs in truth: {'T_liq': 0, 'T_sol': 0, 'sch_range99': 0, 'beta_900': 0, 'kappa_fe_700': 0, 'kappa_nial_700': 0, 'sol1_kappa_fe': 0, 'sol1_fcc_feni': 0, 'a700_NI': 0, 'a700_AL': 0, 'a700_FE': 0, 'a700_MN': 0, 'a900_NI': 0, 'a900_AL': 0, 'a900_FE': 0, 'a900_MN': 0}


In [3]:
import asyncio
from e14c.web import fetch_paper, cite_doi
DOIS=['10.2172/6255887','10.26190/unsworks/21760','10.3390/ma11091777','10.1016/0010-938X(73)90007-3']
async def getpaper(d):
 try:
  bib=await cite_doi(d)
  data=await fetch_paper(doi=d)
  (OUT/(d.split('/')[-1].replace('(','').replace(')','')+'.json')).write_text(json.dumps(data))
  print(d, bib[:160],str(data)[:180])
  return d,data
 except Exception as e:
  print(d,type(e).__name__,str(e)[:200]); return d,None
papers=dict(await asyncio.gather(*(getpaper(d) for d in DOIS)))

10.2172/6255887 KeyError 'PROJECT_ID'
10.1016/0010-938X(73)90007-3 KeyError 'PROJECT_ID'
10.3390/ma11091777 KeyError 'PROJECT_ID'


10.26190/unsworks/21760 KeyError 'PROJECT_ID'


In [4]:
print({k:v for k,v in os.environ.items() if any(x in k for x in ['PROJECT','PERSONA','TRAJECT','WORKSPACE'])})
import inspect
from e14c.web import save_citation
print(inspect.getsource(save_citation))

{'WORKSPACE_DIR': '/workspace', 'GCP_VERTEX_PROJECT_ID': 'crow-prod-451317', 'SANDBOX_WORKSPACE_SIZE': '50Gi', 'TRAJECTORY_ID': '92e929a9-38ce-45ff-855f-de856bce8435', 'WORKSPACE_PATH': '/workspace'}
@auto_inject_docs
def save_citation(bibtex: str) -> Path:
    """Append a BibTeX entry for a web source to this agent's citations file.

    Each agent writes only to
    ``$WORKSPACE_PATH/citations/$PROJECT_ID/$TRAJECTORY_ID.bib``, so agents
    running in parallel never share a file. In chat sessions heron sets
    ``TRAJECTORY_ID`` to the session ID, so an orchestrator keeps one file
    across turns. ``citation_files`` lists every file in the project.

    Example:
        save_citation(bibtex)  # from doi.org, or written by hand

    Returns:
        The path of the file the entry was appended to.
    """
    if not _BIBTEX_ENTRY.match(bibtex):
        raise ValueError(f"Not a BibTeX entry with a cite key: {bibtex[:80]!r}")
    path = _project_citations_dir() / f"{os.environ['TRAJECTO

In [5]:
# The invocation supplies no project ID. Use a clearly local citation namespace only.
os.environ['PROJECT_ID']='local-physics-audit'
papers=dict(await asyncio.gather(*(getpaper(d) for d in DOIS)))
import contextlib, io
buf=io.StringIO()
with contextlib.redirect_stdout(buf): CC.main()
(OUT/'reproduced_calibration.txt').write_text(buf.getvalue())
print(buf.getvalue())
TP.CAL.update(BASE)

10.1016/0010-938X(73)90007-3 HTTPStatusError Client error '403 Forbidden' for url 'https://prod.api.paperqa.app/api/paper'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403


10.2172/6255887 @book{Lewis_1981, title={Evaluation of copper-nickel alloys for OTEC applications}, url={http://dx.doi.org/10.2172/6255887}, DOI={10.2172/6255887}, institution= {'schema_version': 'paperqa.parsing.v1', 'input': {'kind': 'doi', 'value': '10.2172/6255887'}, 'resolved': {'paper_id': 'd66d4818ee7997e3', 'doi': '10.2172/6255887', 'pmc_id': None


10.26190/unsworks/21760 @phdthesis{https://doi.org/10.26190/unsworks/21760,
  doi = {10.26190/UNSWORKS/21760},
  url = {http://hdl.handle.net/1959.4/65594},
  author = {Yang, Yongchao} {'schema_version': 'paperqa.parsing.v1', 'input': {'kind': 'doi', 'value': '10.26190/unsworks/21760'}, 'resolved': {'paper_id': '0f07033076000ba5', 'doi': '10.26190/unsworks/21760'


10.3390/ma11091777 @article{Yang_2018, title={Effect of Al Element on the Microstructure and Properties of Cu-Ni-Fe-Mn Alloys}, volume={11}, ISSN={1996-1944}, url={http://dx.doi.o {'schema_version': 'paperqa.parsing.v1', 'input': {'kind': 'doi', 'value': '10.3390/ma11091777'}, 'resolved': {'paper_id': 'bfa5c500dfa2adee', 'doi': '10.3390/ma11091777', 'pmc_id'


loss 0.5951
{'a_al': 0.05, 'a_ni0': 0.1003, 'a_ni': 0.513, 'a_fe': 0.5773, 'tau': np.float64(10.0), 'K_cu': 100000.0, 'i_fl': 3.604} (tau in days)
  90-10 / 70-30, 30 d                      model    1.241  target    1.400
  Cu / 70-30, 30 d                         model    3.469  target    3.000
  Al bronze (C61400) / 70-30, 30 d         model    1.818  target    2.100
  Cu-10Ni-0Fe / Cu-10Ni-1.5Fe, 60 d        model    2.597  target    3.100
  Cu rate days 16-78 / days 0-5            model    0.876  target    0.700
  Cu20Ni rate days 16-78 / days 0-5        model    0.340  target    0.360
  Cu30Ni rate days 16-78 / days 0-5        model    0.250  target    0.180
  Cu i(2 h), uA/cm2                        model    6.026  target    7.897
  Cu9.3Ni i(2 h), uA/cm2                   model    5.685  target    4.701
  Cu18.8Ni i(2 h), uA/cm2                  model    5.359  target    3.778
  Cu28.4Ni i(2 h), uA/cm2                  model    5.049  target    2.479
  Cu7Al i(2 h), uA/cm2      

In [6]:
for d,p in papers.items():
 if p:
  print(d,p.keys(), 'pages',len(p.get('pages',[])))
  text='\n'.join(pg.get('text','') for pg in p.get('pages',[]))
  (OUT/(d.split('/')[-1]+'.txt')).write_text(text)
  if '1777' in d:
   print(text[:4000])
print('Grid sol1 phase medians/max\n',df.filter(regex='^sol1_').agg(['min','median','max']))
print('Excluded grid points:',{'liq':sum(df.T_liq>1250),'cu':sum(df.Cu<65)})

10.2172/6255887 dict_keys(['schema_version', 'input', 'resolved', 'open_access', 'parsing', 'metadata', 'pages', 'media']) pages 103
10.26190/unsworks/21760 dict_keys(['schema_version', 'input', 'resolved', 'open_access', 'parsing', 'metadata', 'pages', 'media']) pages 137
10.3390/ma11091777 dict_keys(['schema_version', 'input', 'resolved', 'open_access', 'parsing', 'metadata', 'pages', 'media']) pages 11
    materials



Article
Effect of Al Element on the Microstructure and
Properties of Cu-Ni-Fe-Mn Alloys

Ran Yang 1, Jiuba Wen 1,2,*, Yanjun Zhou 1,2, Kexing Song 1,2,* and Zhengcheng Song 1

  1   School of Materials Science and Engineering, Henan University of Science and Technology,
    Luoyang 471023, China; <email removed> (R.Y.); <email removed> (Y.Z.); <email removed> (Z.S.)
  2   Collaborative Innovation Center of Nonferrous Metals, Henan University of Science and Technology,
    Luoyang 471023, China
 *   Correspondence: <email removed> (J.W.); <email removed> (K.S.)


 Rece

In [7]:
text=(OUT/'ma11091777.txt').read_text()
for term in ['10 days','Table 3','Table 4','0.0215','240','30 min','immers']:
 pos=0
 print('\nTERM',term)
 for _ in range(4):
  pos=text.lower().find(term.lower(),pos)
  if pos<0: break
  print(text[max(0,pos-250):pos+750]);pos+=len(term)
# inspect Lewis relevant exact compositions and late/early sentence
lt=(OUT/'6255887.txt').read_text()
for term in ['0.36','38.4','0.0023','78 days']:
 pos=lt.find(term); print('\nLewis',term,lt[max(0,pos-350):pos+900])


TERM 10 days
e immersion corrosion, and the quality was accurately determined
by an analytical balance.
    The corrosion medium was artiﬁcial seawater; the artiﬁcial seawater composition is shown in
Table 2 [15]. The test was carried out at room temperature for 10 days, and the corrosion medium was
Materials 2018, 11, 1777                                                                                    3 of 11



replaced every seventh day. After completion of the test, with the volume ratio of 1:2 hydrochloric
acid solution, on the corrosion after cleaning, the corrosion products were removed after cleaning with
alcohol, and then the sample was put in a drying oven drying to calculate the quality loss of samples
before and after corrosion. The corrosion rate R was calculated by the change of sample quality before
and after corrosion. Then the corrosion rate was calculated according to Equation (1):

                                          8.76 × 107 × (M −M1)
                   

In [8]:
from scipy.interpolate import RegularGridInterpolator
fields=TP.Grid.FIELDS
V=np.stack([TP.grid().V[k].numpy() for k in fields],axis=-1)
interp=RegularGridInterpolator([TP.AXES[k] for k in ['NI','AL','FE','MN']],V,bounds_error=False,fill_value=None)
FIDX={k:i for i,k in enumerate(fields)}
maxwt=TP.WT_MAX.numpy()
def np_features(wt,mode='900',ni_shape='linear'):
 wt=np.atleast_2d(wt); vals=interp(wt)
 if mode=='nominal': a=wt.copy()
 else:
  a=np.minimum(wt,np.maximum(vals[:,[FIDX['a'+mode+'_'+e] for e in ['NI','AL','FE','MN']]],0))
 ni=a[:,0]/30
 if ni_shape=='sat10': ni=(1-np.exp(-a[:,0]/10))/(1-np.exp(-3))
 if ni_shape=='sat5': ni=(1-np.exp(-a[:,0]/5))/(1-np.exp(-6))
 return np.array([a[:,1]/7,ni,np.minimum(a[:,2],1.5)/1.5]).T
# Match calibration in fast numpy evaluations; no statistical uncertainty is implied by residual weights.
alloys=CC.ALLOY.copy(); alloys['Yang2018']=(7,3,1,1)
kn=list(alloys); kid={k:i for i,k in enumerate(kn)}
TAV={}
def av_vector(p,t0,t1,features):
 t0=max(t0,1);t=np.logspace(np.log10(t0),np.log10(t1),200)
 fast=p[0]*features[:,0]+p[1]*features[:,1];slow=p[2]*features[:,1]+p[3]*features[:,2]
 q=10**(-fast[:,None]-slow[:,None]*(1-np.exp(-t/p[4])))
 cur=1/(1/30+1/np.sqrt(p[5]*q/(2*t)))+p[6]*q
 return np.trapezoid(cur,t,axis=-1)/(t1-t0)
def predictions(p,features):
 m=av_vector(p,0,30*CC.DAY,features); a=av_vector(p,0,60*CC.DAY,features)
 e=av_vector(p,0,5*CC.DAY,features);l=av_vector(p,16*CC.DAY,78*CC.DAY,features)
 get=lambda arr,k: arr[kid[k]]
 f=p[0]*features[:,0]+p[1]*features[:,1];s=p[2]*features[:,1]+p[3]*features[:,2]
 q=10**(-f-s*(1-np.exp(-7200/p[4])))
 i=1/(1/30+1/np.sqrt(p[5]*q/(2*7200)))+p[6]*q
 return np.array([get(m,'C70600')/get(m,'C71500'),get(m,'Cu')/get(m,'C71500'),get(m,'C61400')/get(m,'C71500'), get(a,'Cu10Ni0Fe')/get(a,'Cu10Ni1.5Fe'),*[get(l,k)/get(e,k) for k in ['Cu','Cu20Ni','Cu30Ni']],*[get(i,k) for k in ['Cu','Cu9.3Ni','Cu18.8Ni','Cu28.4Ni','Cu7Al']],get(m,'C71500')])
target=np.array([1.4,3,2.1,3.1,.7,.36,.18,92.4/11.7,55/11.7,44.2/11.7,29/11.7,37/11.7,2])
weights=np.array([1,1,1,1,.7,.7,.7,.5,.5,.5,.5,.5,.7])
lo=np.array([.05,.05,.05,.05,.25*CC.DAY,1e5,.05]);hi=np.array([1.5]*4+[10*CC.DAY,1e9,10])
pbase=np.array([BASE[k] for k in CC.NAMES])
feat=np_features(list(alloys.values()))
print('np vs supplied residual max error',np.max(np.abs(weights*np.log(predictions(pbase,feat)/target)-CC.residuals(pbase,CC.terms()))))
def fit_model(label,mode='900',shape='linear',target_=None,weights_=None,lo_=None,hi_=None,fix=None,yang=False):
 ft=np_features(list(alloys.values()),mode,shape)
 tar=target.copy() if target_ is None else target_; w=weights.copy() if weights_ is None else weights_
 lower=lo.copy() if lo_ is None else lo_; upper=hi.copy() if hi_ is None else hi_
 fix=fix or {}; free=[i for i in range(7) if i not in fix]
 def unpack(z):
  p=pbase.copy();p[free]=np.exp(z)
  for i,v in fix.items():p[i]=v
  return p
 def res(z):
  p=unpack(z);r=w*np.log(predictions(p,ft)/tar)
  if yang:
   a=av_vector(p,0,10*CC.DAY,ft);r=np.r_[r,np.log(a[kid['Yang2018']]/a[kid['C70600']]/.6)]
  return r
 best=None
 for start in [pbase,np.array(CC.X0)]:
  z=np.log(np.clip(start[free],lower[free]*1.00001,upper[free]/1.00001))
  rr=least_squares(res,z,bounds=(np.log(lower[free]),np.log(upper[free])),max_nfev=700,ftol=1e-10,xtol=1e-10,gtol=1e-9)
  if best is None or sum(rr.fun**2)<sum(best.fun**2):best=rr
 return {'label':label,'p':unpack(best.x),'loss':sum(best.fun**2),'mode':mode,'shape':shape,'pred':predictions(unpack(best.x),ft)}
fbase=fit_model('Original targets/bounds')
print(fbase)

np vs supplied residual max error 4.440892098500626e-16
{'label': 'Original targets/bounds', 'p': array([5.00000000e-02, 1.00327608e-01, 5.13020162e-01, 5.77273726e-01,
       8.64000000e+05, 1.00000000e+05, 3.60366951e+00]), 'loss': np.float64(0.5950821361621409), 'mode': '900', 'shape': 'linear', 'pred': array([1.24110498, 3.46864584, 1.81816237, 2.59687045, 0.87580215,
       0.33960022, 0.25039253, 6.02611109, 5.68505329, 5.35852359,
       5.04948818, 5.50908304, 1.11703774])}


In [9]:
from scipy.special import expit
TT=TP.TGRID.numpy()
def np_outputs(X,p=pbase,mode='900',shape='linear',cpor=1,isel=5,knuc=1):
 X=np.atleast_2d(X);wt=X[:,:4]*maxwt;v=interp(wt)
 get=lambda f:v[:,FIDX[f]]
 a7=v[:,[FIDX['a700_'+e] for e in ['NI','AL','FE','MN']]]
 a9=v[:,[FIDX['a900_'+e] for e in ['NI','AL','FE','MN']]]
 f=np.clip(get('sch_range99')/250,0,1);g=np.clip((get('sol1_kappa_fe')+get('sol1_fcc_feni'))/.05,0,1)
 moles=a9/np.array([TP.MW[e] for e in ['NI','AL','FE','MN']]);cu=(100-a9.sum(axis=-1))/TP.MW['CU']
 at=100*moles/(moles.sum(axis=-1)+cu)[:,None]
 conductivity=732/(1.7+at@np.array([1.1,.65,2,2.9]))+5
 h=.95*conductivity/TP.K_CU*(get('T_sol')-25)/(1084.6-25)
 q=np.exp(-((X[:,4]-(.35+.4*f))/.28)**2)*(1-.35*f*f)*(1-.1*X[:,3])
 P=X[:,5];vel=X[:,6];u=P-vel-(.05+.25*f)
 window=np.exp(-(u/.2)**2)*(1-.35*expit((P-.85)/.05))
 feff=f*(1-knuc*.85*g/(g+.12))
 rho=window*q**.3*(1-.75*expit((feff-.8)/.05))*(1-expit((h-P)/.08))
 beta=np.clip(get('beta_900'),0,1)*expit((vel-P+.2)/.15)
 feat=np_features(wt,mode,shape);fast=p[0]*feat[:,0]+p[1]*feat[:,1];slow=p[2]*feat[:,1]+p[3]*feat[:,2]
 film=10**(-fast[:,None]-slow[:,None]*(1-np.exp(-TT/p[4])))
 current=1/(1/30+1/np.sqrt(p[5]*film/(2*TT)))+p[6]*film
 M=(np.trapezoid(current,TT,axis=-1)/CC.DAY+30/CC.DAY+beta*isel*29)*(1+cpor*(1-rho))
 feas=(wt.sum(axis=-1)<=35+1e-9)&(get('T_liq')<=1250)&((wt[:,1]<=7)|(wt[:,0]>=wt[:,2]))&((wt[:,1]>=1.5)|(wt[:,2]<=2.3))
 return dict(M=M,rho=rho,beta=beta,feas=feas,f=f,g=g,h=h)
# numpy truth agreement
rng=np.random.default_rng(44);test=rng.random((100,7)); xx=torch.tensor(test)
TP.CAL.update(BASE);oo=TP.PhysTruth(m_ref=0).full(xx);nn=np_outputs(test)
for k in ['rho','beta','f','g','h']:
 print(k,np.max(abs(nn[k]-oo['rho_m' if k=='rho' else k].numpy())))
fast,slow=TP.film_terms(xx[:,:4]);print('M error',max(abs(nn['M']-TP.mass_loss_30d(fast,slow,oo['beta'],oo['rho_m']).numpy())))
# For beta-free refs, maximize density once; reused for refit bars (same convention as supplied).
refx={};refrho={}
for alloy in ['C71500','C70600']:
 mr,rho,x=TP.reference_m_for(alloy);refx[alloy]=x.numpy();refrho[alloy]=rho
 print(alloy,'reference wt',x[:4]*TP.WT_MAX,'rho',rho,'M',np_outputs(x.numpy())['M'][0], 'm',mr)
# Ni floor: constrained 7-D optimization, exact mass equality, all proposal constraints.
# a maximizes q analytically, leaving 6 independent variables (4 wt%, P, v).
def candidate(z):
 wt=np.atleast_2d(z)[...,:4];vs=interp(wt);f=np.clip(vs[:,FIDX['sch_range99']]/250,0,1)
 return np.column_stack([wt/maxwt,.35+.4*f,np.atleast_2d(z)[:,4:6]])
bounds=[(0,30),(0,9.5),(0,4.5),(0,4),(0,1),(0,1)]
def floor_for(model,alloy,bar='deposited',extra=None,seed=11):
 p=model['p'];mode=model['mode'];shape=model['shape'];extra=extra or {}
 barX=refx[alloy].copy()
 Mr=np_outputs(barX,p,mode,shape,**extra)['M'][0]
 if bar=='wrought':
  rho=np_outputs(barX,p,mode,shape,**extra)['rho'][0]
  Mr=Mr/(1+extra.get('cpor',1)*(1-rho))
 def con(z):
  z=np.atleast_2d(z);X=candidate(z);o=np_outputs(X,p,mode,shape,**extra);vals=interp(z[:,:4])
  return np.column_stack([1-o['M']/Mr,(35-z[:,:4].sum(axis=1))/10,(1250-vals[:,FIDX['T_liq']])/100,np.where(z[:,1]>7,(z[:,0]-z[:,2])/5,1),np.where(z[:,1]<1.5,(2.3-z[:,2])/3,1)])
 def obj(z):
  co=con(z)[0];return z[0]+1e4*np.minimum(co,0)@np.minimum(co,0)
 de=differential_evolution(obj,bounds,seed=seed,popsize=10,maxiter=100,tol=1e-6,polish=False)
 starts=[de.x,np.array([15,3,1.5,0,.58,.48]),np.array([10,1.5,2.3,0,.55,.45]),np.array([0,7,4.5,0,.6,.5])]
 sols=[]
 for z0 in starts:
  r=minimize(lambda z:z[0],z0,bounds=bounds,constraints=[{'type':'ineq','fun':lambda z:con(z)[0]}],method='SLSQP',options={'maxiter':220,'ftol':2e-9})
  if min(con(r.x)[0])>=-1e-6:sols.append(r)
 if not sols:return {'floor':None,'barM':Mr}
 best=min(sols,key=lambda r:r.fun);z=best.x
 return {'floor':z[0],'wt':z[:4].tolist(),'P':z[4],'v':z[5],'barM':Mr,'M':np_outputs(candidate(z),p,mode,shape,**extra)['M'][0],'success':best.success}
t0=time.time()
floors_base={a:floor_for(fbase,a) for a in refx}
print('BASE floors',floors_base,'seconds',time.time()-t0)

rho 4.440892098500626e-16
beta 5.551115123125783e-17
f 3.3306690738754696e-16
g 2.220446049250313e-16
h 8.326672684688674e-17
M error 5.684341886080802e-14


C71500 reference wt tensor([30.0000,  0.0000,  0.6750,  0.8000]) rho 0.9076691548142546 M 36.71241189759363 m 0.8517398423396847
C70600 reference wt tensor([9.9900, 0.0000, 1.3950, 0.8000]) rho 0.9775587239470636 M 42.73625502647318 m 0.8077508123376581


BASE floors {'C71500': {'floor': np.float64(12.506956357913271), 'wt': [12.506956357913271, 1.4999998587516645, 1.5000000010014376, 2.4664478600759418e-12], 'P': np.float64(0.5744993723522165), 'v': np.float64(0.4794628978802868), 'barM': np.float64(36.67793929285784), 'M': np.float64(36.67793930373354), 'success': True}, 'C70600': {'floor': np.float64(6.400801931242165), 'wt': [6.400801931242165, 2.999999999394912, 1.5000000712402572, 5.546858853845616e-13], 'P': np.float64(0.5828066733327536), 'v': np.float64(0.5134430603982755), 'barM': np.float64(42.709724315907366), 'M': np.float64(42.70972431779683), 'success': True}} seconds 28.89794421195984


In [10]:
models=[fbase]
for name,kw in [
 ('90/70 target 0.95 (actual 30 d)',{'target_':np.r_[.95,target[1:]]}),
 ('90/70 target 2.3 (62 d alternative)',{'target_':np.r_[2.3,target[1:]]}),
 ('Drop invented Cu ratio and unidentified 1-y bronze',{'weights_':weights*np.array([1,0,0,1,1,1,1,1,1,1,1,1,1])}),
 ('Equal log-residual weights',{'weights_':np.ones(13)}),
 ('2-h measurements full weight; omit 1-y bronze',{'weights_':np.array([1,0,0,1,.7,.7,.7,1,1,1,1,1,.7])}),
 ('Allow a_Al down to near zero',{'lo_':np.r_[1e-6,lo[1:]]}),
 ('Allow tau up to 100 d',{'hi_':np.r_[hi[:4],100*CC.DAY,hi[5:]]}),
 ('Allow K down to 1',{'lo_':np.r_[lo[:5],1,lo[6]]}),
 ('Relax all three active bounds',{'lo_':np.array([1e-6,.001,.001,.001,.05*CC.DAY,1,.001]),'hi_':np.array([3,3,3,3,100*CC.DAY,1e9,30])}),
 ('Fix a_Al = 0.4; refit others',{'fix':{0:.4}}),
 ('Fix tau = 3 d; refit others',{'fix':{4:3*CC.DAY}}),
 ('Fix K = 1e7; refit others',{'fix':{5:1e7}}),
 ('Add Yang 2018 ratio 0.60 at 10 d',{'yang':True}),
 ('Ni benefit saturating, scale 10 wt%',{'shape':'sat10'}),
 ('Use 700 C matrix; refit',{'mode':'700'}),
 ('Use nominal composition; refit',{'mode':'nominal'}),
]:
 m=fit_model(name,**kw);models.append(m)
 print(name,'loss',round(m['loss'],4),'params',np.round(m['p']/np.array([1,1,1,1,CC.DAY,1,1]),4))
# Save fits first, then assess floors so all intermediate work survives.
def jsondefault(x):
 if isinstance(x,np.ndarray):return x.tolist()
 if isinstance(x,np.generic):return x.item()
 raise TypeError(type(x).__name__)
(OUT/'refits.json').write_text(json.dumps(models,default=jsondefault,indent=2))

90/70 target 0.95 (actual 30 d) loss 0.6129 params [5.0000e-02 7.0800e-02 4.8780e-01 6.4630e-01 1.0000e+01 1.0000e+05
 3.4429e+00]
90/70 target 2.3 (62 d alternative) loss 0.8722 params [1.3720e-01 1.6850e-01 5.5120e-01 4.7340e-01 1.0000e+01 1.0000e+05
 4.1372e+00]
Drop invented Cu ratio and unidentified 1-y bronze loss 0.4434 params [4.502e-01 1.847e-01 4.917e-01 6.490e-01 1.000e+01 1.000e+05 4.712e+00]


Equal log-residual weights loss 1.4889 params [1.588e-01 1.716e-01 4.648e-01 5.416e-01 1.000e+01 1.000e+05 3.428e+00]
2-h measurements full weight; omit 1-y bronze loss 0.7501 params [4.2310e-01 3.5940e-01 3.3950e-01 6.7930e-01 1.0000e+01 1.0000e+05
 4.3083e+00]


Allow a_Al down to near zero loss 0.5938 params [2.9900e-02 9.3400e-02 5.1120e-01 5.8130e-01 1.0000e+01 1.0000e+05
 3.5376e+00]
Allow tau up to 100 d loss 0.3982 params [5.30000e-02 2.60800e-01 7.20200e-01 1.50000e+00 5.66917e+01 1.00000e+05
 3.91880e+00]


Allow K down to 1 loss 0.4567 params [ 0.05    0.05    0.5976  0.5679 10.      1.      5.0524]
Relax all three active bounds loss 0.2586 params [5.3400e-02 2.3790e-01 1.3036e+00 2.4146e+00 1.0000e+02 1.0000e+00
 5.5173e+00]


Fix a_Al = 0.4; refit others loss 1.0343 params [4.000e-01 2.191e-01 5.447e-01 5.113e-01 1.000e+01 1.000e+05 4.801e+00]
Fix tau = 3 d; refit others loss 1.0151 params [5.00000000e-02 5.00000000e-02 4.50600000e-01 4.66000000e-01
 3.00000000e+00 1.25686528e+05 3.14620000e+00]


Fix K = 1e7; refit others loss 2.165 params [2.5260e-01 6.8340e-01 5.0000e-02 7.6680e-01 1.0000e+01 1.0000e+07
 2.4987e+00]
Add Yang 2018 ratio 0.60 at 10 d loss 0.9391 params [1.6960e-01 1.2020e-01 5.4630e-01 5.2360e-01 1.0000e+01 1.0000e+05
 3.9037e+00]


Ni benefit saturating, scale 10 wt% loss 0.7493 params [6.620e-02 1.565e-01 4.478e-01 5.462e-01 1.000e+01 1.000e+05 3.922e+00]
Use 700 C matrix; refit loss 0.6591 params [7.0500e-02 5.0000e-02 3.6460e-01 1.5000e+00 1.0000e+01 1.0000e+05
 3.3284e+00]


Use nominal composition; refit loss 0.7001 params [5.0000e-02 1.2740e-01 5.3580e-01 5.2070e-01 1.0000e+01 1.0000e+05
 3.7632e+00]


11479

In [11]:
t0=time.time()
floor_results=[]
for m in models:
 rr={'label':m['label'],'loss':m['loss']}
 for a in refx:
  rr[a]=floor_for(m,a)
 floor_results.append(rr)
 (OUT/'floor_sensitivities.json').write_text(json.dumps(floor_results,default=jsondefault,indent=2))
 print(m['label'], [(a,round(rr[a]['floor'],3) if rr[a]['floor'] is not None else None) for a in refx],flush=True)
print('Elapsed',time.time()-t0)

Original targets/bounds [('C71500', np.float64(12.507)), ('C70600', np.float64(6.401))]


90/70 target 0.95 (actual 30 d) [('C71500', np.float64(7.388)), ('C70600', np.float64(5.961))]


90/70 target 2.3 (62 d alternative) [('C71500', np.float64(16.729)), ('C70600', np.float64(5.31))]


Drop invented Cu ratio and unidentified 1-y bronze [('C71500', np.float64(0.0)), ('C70600', np.float64(0.0))]


Equal log-residual weights [('C71500', np.float64(13.468)), ('C70600', np.float64(4.399))]


...

Add Yang 2018 ratio 0.60 at 10 d [('C71500', np.float64(13.167)), ('C70600', np.float64(4.135))]


Ni benefit saturating, scale 10 wt% [('C71500', np.float64(5.562)), ('C70600', np.float64(6.96))]


Use 700 C matrix; refit [('C71500', np.float64(16.82)), ('C70600', np.float64(7.629))]


Use nominal composition; refit [('C71500', np.float64(12.856)), ('C70600', np.float64(4.081))]


Elapsed 410.0679247379303


In [12]:
src=Path('check_phys.py').read_text()
# Re-run all available checks; explicitly omit seed-design statistic, since BA.rand_designs was not uploaded.
src=src.replace('import bofn_alloy as BA  # noqa: E402', 'import types\nimport truth_phys as TP\nBA = types.SimpleNamespace(feasible_designs=TP.feasible)')
src=src.replace('from physics import truth_phys as TP  # noqa: E402','import truth_phys as TP')
src=src.replace('HERE = Path(__file__).parent','HERE = Path(__file__).parent\nsys.path.insert(0, "/workspace")')
src=src.replace('torch.set_default_dtype(torch.double)','torch.set_default_dtype(torch.double)\ntorch.set_num_threads(4)')
a=src.index('    seeds = []');b=src.index('    lines += [f"- Expected best',a)
src=src[:a]+'    res["landscape"]["seed_designs_best_mean"] = float("nan")  # Not reproducible: bofn_alloy absent.\n'+src[b:]
(OUT/'check_phys_adapter.py').write_text(src)
logs={}
for ref in ['C71500','C70600']:
 env=os.environ.copy();env['BOFN_REF']=ref
 p=subprocess.run([sys.executable,str(OUT/'check_phys_adapter.py')],env=env,capture_output=True,text=True,timeout=500)
 (OUT/f'check_{ref}.log').write_text(p.stdout+p.stderr)
 print(ref,'return',p.returncode,p.stdout[-1600:],p.stderr[-500:])
 if p.returncode==0:
  suf='' if ref=='C71500' else '_c70600'
  logs[ref]=json.loads((OUT/f'phys_tables{suf}.json').read_text())

C71500 return 0 de the proposal's limits (liquidus <= 1,250 C, Cu >= 65 wt%, Ni >= Fe above 7 wt% Al, Fe <= 2.3 wt% below 1.5 wt% Al).
- Pass immersion: 0.2% of the box, 0.1% of the feasible space.
- J > 0: 0.0% of the box, 0.0% of the feasible space.
- Within 60% / 80% of the optimum (feasible space): 0.00% / 0.000%.
- Lowest Ni among designs that pass immersion: 14.64 wt% (box), 14.64 wt% (feasible).

- With a 10 wt% Ni floor (46% of the box): 0.2% pass immersion, 0.0% have J > 0, 0.00% are within 60% of the optimum.
  - Ni 0-1 wt%: 0.00% of feasible designs pass immersion
  - Ni 1-2 wt%: 0.00% of feasible designs pass immersion
  - Ni 2-3 wt%: 0.00% of feasible designs pass immersion
  - Ni 3-4 wt%: 0.00% of feasible designs pass immersion
  - Ni 4-5 wt%: 0.00% of feasible designs pass immersion
  - Ni 5-6 wt%: 0.00% of feasible designs pass immersion
  - Ni 6-7 wt%: 0.00% of feasible designs pass immersion
  - Ni 8-10 wt%: 0.00% of feasible designs pass immersion
  - Ni 10-12 wt%: 

C70600 return 0 side the proposal's limits (liquidus <= 1,250 C, Cu >= 65 wt%, Ni >= Fe above 7 wt% Al, Fe <= 2.3 wt% below 1.5 wt% Al).
- Pass immersion: 0.7% of the box, 0.7% of the feasible space.
- J > 0: 0.2% of the box, 0.2% of the feasible space.
- Within 60% / 80% of the optimum (feasible space): 0.00% / 0.000%.
- Lowest Ni among designs that pass immersion: 9.20 wt% (box), 9.20 wt% (feasible).

- With a 10 wt% Ni floor (46% of the box): 1.1% pass immersion, 0.3% have J > 0, 0.01% are within 60% of the optimum.
  - Ni 0-1 wt%: 0.00% of feasible designs pass immersion
  - Ni 1-2 wt%: 0.00% of feasible designs pass immersion
  - Ni 2-3 wt%: 0.00% of feasible designs pass immersion
  - Ni 3-4 wt%: 0.00% of feasible designs pass immersion
  - Ni 4-5 wt%: 0.00% of feasible designs pass immersion
  - Ni 5-6 wt%: 0.00% of feasible designs pass immersion
  - Ni 6-7 wt%: 0.00% of feasible designs pass immersion
  - Ni 8-10 wt%: 0.02% of feasible designs pass immersion
  - Ni 10-12 wt%: 

In [13]:
def robust_floor(model,alloy,bar='deposited',extra=None,seed=73):
 p=model['p'];mode=model['mode'];shape=model['shape'];extra=extra or {}
 oo=np_outputs(refx[alloy],p,mode,shape,**extra); Mr=oo['M'][0]
 if bar=='wrought':Mr/=1+extra.get('cpor',1)*(1-oo['rho'][0])
 best=None
 for region in range(3):
  bb=bounds.copy()
  bb[1]=[(0,1.5),(1.5,7),(7,9.5)][region]
  if region==0:bb[2]=(0,2.3)
  def con(z):
   z=np.atleast_2d(z);o=np_outputs(candidate(z),p,mode,shape,**extra);vals=interp(z[:,:4])
   return np.column_stack([1-o['M']/Mr,(35-z[:,:4].sum(axis=1))/10,(1250-vals[:,FIDX['T_liq']])/100, (z[:,0]-z[:,2])/5 if region==2 else np.ones(len(z))])
  def obj(z):
   zs=np.atleast_2d(z.T) if np.ndim(z)==2 else np.atleast_2d(z)
   co=con(zs);a=zs[:,0]+1e5*np.square(np.minimum(co,0)).sum(axis=1)
   return a if np.ndim(z)==2 else a[0]
  de=differential_evolution(obj,bb,seed=seed,popsize=12,maxiter=170,tol=1e-8,polish=False,vectorized=True,updating='deferred')
  starts=[de.x,np.clip([15,3,1.5,0,.58,.48],*np.array(bb).T),np.clip([0,7,4.5,0,.6,.5],*np.array(bb).T)]
  for zz in starts:
   rr=minimize(lambda z:z[0],zz,bounds=bb,constraints=[{'type':'ineq','fun':lambda z:con(z)[0]}],method='SLSQP',options={'maxiter':250,'ftol':1e-10})
   if min(con(rr.x)[0])>=-1e-7 and (best is None or rr.fun<best.fun):best=rr
 if best is None:return {'floor':None,'barM':Mr}
 z=best.x; XX=candidate(z);oo=np_outputs(XX,p,mode,shape,**extra)
 return {'floor':z[0],'wt':z[:4].tolist(),'P':z[4],'v':z[5],'x':XX[0].tolist(),'barM':Mr,'M':oo['M'][0],'rho':oo['rho'][0],'beta':oo['beta'][0],'success':best.success}
t0=time.time()
robust_results=[]
for m in models:
 rr={'label':m['label'],'loss':m['loss']}
 for a in refx:rr[a]=robust_floor(m,a)
 robust_results.append(rr)
 (OUT/'floor_sensitivities_verified.json').write_text(json.dumps(robust_results,default=jsondefault,indent=2))
 print(m['label'],[(a,rr[a]['floor']) for a in refx],flush=True)
print('Elapsed',time.time()-t0)

Original targets/bounds [('C71500', np.float64(12.506955518033758)), ('C70600', np.float64(6.4008018892363205))]


90/70 target 0.95 (actual 30 d) [('C71500', np.float64(7.388080188908324)), ('C70600', np.float64(5.961349027601675))]


90/70 target 2.3 (62 d alternative) [('C71500', np.float64(16.72900633386296)), ('C70600', np.float64(5.31029539890719))]


Drop invented Cu ratio and unidentified 1-y bronze [('C71500', np.float64(0.0)), ('C70600', np.float64(0.0))]


Equal log-residual weights [('C71500', np.float64(13.468301524829082)), ('C70600', np.float64(4.399191593729746))]


...

Add Yang 2018 ratio 0.60 at 10 d [('C71500', np.float64(13.166631640207287)), ('C70600', np.float64(4.135140361119429))]


Ni benefit saturating, scale 10 wt% [('C71500', np.float64(5.561648418682869)), ('C70600', np.float64(6.960050314804733))]


Use 700 C matrix; refit [('C71500', np.float64(1.3032278838511906)), ('C70600', np.float64(0.9498917281840701))]


Use nominal composition; refit [('C71500', np.float64(12.856156362037773)), ('C70600', np.float64(4.081080864000575))]


Elapsed 541.1489679813385


In [14]:
for term in ['40%','40 %','C70600','commercial','0.035','2. Experimental']:
 print('\n###',term)
 pos=0
 while True:
  pos=text.lower().find(term.lower(),pos)
  if pos<0:break
  print(text[max(0,pos-180):pos+450]);pos+=len(term)
print('\nVerified floor witnesses:')
for i in [0,3,14,15]:
 print(robust_results[i])
# Primary thesis 2h rates and Faraday conversion
th=(OUT/'21760.txt').read_text()
for term in ['92.4','0.4','11.7','Table 4.','valence']:
 j=th.find(term);print('\nTHESIS',term,th[max(0,j-200):j+700])


### 40%
strength, 395 MPa. It was notable that, with the addition of aluminum content,
the elongation of Cu-7Ni-xAl-1Fe-1Mn alloys ﬁrstly increases and then decreases, but still more than
40% higher than without aluminum content. The addition of aluminum element did not cause a severe
impairment of tensile ductility.


                                   420                                                                 100
                                   400                                    Tensile strength
                                   380                                  Elongation         90

                        

### 40 %

### C70600
 alloy. Moreover, the corrosion rate in artiﬁcial
 seawater of Cu-7Ni-3Al-1Fe-1Mn alloy is 0.0215 mm/a which exhibits a better corrosion resistance
 than the commercially used UNS C70600. It is conﬁrmed that the second-phase transformation of
 Cu-7Ni-xAl-1Fe-1Mn alloy follows the sequence of α solid solution →Ni3Al →Ni3Al + NiAl →
 Ni3A

In [15]:
from e14c.web import web_search
res=await web_search('site.copper.org C71500 thermal conductivity electrical conductivity C70600 C61400 C95800',num_results=6)
print([(r['title'],r['url'],r['full_text'][:500]) for r in res])
# Inspect source methods and current density tables to see whether conversion n=2 matches thesis convention.
for term in ['Table 13','Table 14','7.897','55.0','29.0','3270','equivalent weight','0.00327','ASTM','charge']:
 pos=th.find(term)
 print('\nTH',term,th[max(0,pos-160):pos+600] if pos>=0 else 'not found')
print('Witnesses baseline and data omission')
for i in [0,3]:
 for a in refx:print(i,a,robust_results[i][a])

[('Copper.org - C71500 Alloy', 'https://alloys.copper.org/alloy/C71500?referrer=facetedsearch', 'Copper.org - C71500 Alloy\n\nChemical Composition\n\n| | Elements | | | | | |\n| --- | --- | --- | --- | --- | --- | --- |\n| | Cu (1,2) | Pb | Zn | Fe | Ni (3) | Mn |\n| (1) Cu + Sum of Named Elements 99.5% min. (2) Cu value includes Ag. (3) Ni value includes Co. | | | | | | |\n| Min (%) | | | | 0.40 | 29.0 | |\n| Max (%) | Rem | 0.05 | 1.0 | 1.0 | 33.0 | 1.0 |\n\n Mechanical Properties * \n\n| Form | Temper | Temper Code | Tensile Strength (ksi) | YS-0.5% Ext (ksi) | Elongation (%) | Rockwell B scale | Roc'), ('Properties of Wrought and Cast Copper Alloys: Copper Alloys Search - Basic', 'https://www.copper.org/resources/properties/db/basic-search.php?alloy-select-properties=physical&alloys-select=C71500&submit-multiple-alloys=Display', 'Properties of Wrought and Cast Copper Alloys: Copper Alloys Search - Basic\n\n# Properties of Wrought and Cast Copper Alloys\n\n## Copper Alloys Search - 

In [16]:
from e14c.web import scrape,save_citation
async def getcda(a):
 url=f'https://alloys.copper.org/alloy/{a}'
 try:
  s=await scrape(url);(OUT/f'CDA_{a}.md').write_text(s)
  save_citation('@online{cda2026'+a+',\n author={{Copper Development Association}},\n title={'+a+' Alloy},\n url={'+url+'},\n urldate={'+time.strftime('%Y-%m-%d')+'}\n}')
  lines=s.splitlines();print(a,'\n'.join(l for l in lines if any(t in l.lower() for t in ['conductivity','solidus','liquidus'])))
  return a,s
 except Exception as e:print(a,type(e).__name__,str(e)[:200]);return a,None
cda=dict(await asyncio.gather(*(getcda(a) for a in ['C71500','C70600','C61400','C95800'])))
# Capture exact original-CAL witnesses with modest rounding and a passing safety margin.
TP.CAL.update(BASE)
for a,ni,al in [('C71500',12.52,1.5),('C70600',6.42,3)]:
 z=np.array([ni,al,1.5,0,.58,.48 if a=='C71500' else .51]);X=candidate(z)
 oo=TP.PhysTruth(m_ref=0).full(torch.tensor(X))
 fast,slow=TP.film_terms(torch.tensor(X[:,:4]));M=TP.mass_loss_30d(fast,slow,oo['beta'],oo['rho_m'])
 print('Witness',a,'x',X,'feasible',TP.feasible(torch.tensor(X)).item(),'M',M.item(),'rho',oo['rho_m'].item(),'m',oo['m'].item(),'ref',TP.reference_m_for(a)[0])

C95800 HTTPStatusError Failed to scrape https://alloys.copper.org/alloy/C95800: HTTP 503 with description None
C70600 HTTPStatusError Failed to scrape https://alloys.copper.org/alloy/C70600: HTTP 503 with description None
C71500 HTTPStatusError Failed to scrape https://alloys.copper.org/alloy/C71500: HTTP 503 with description None
C61400 HTTPStatusError Failed to scrape https://alloys.copper.org/alloy/C61400: HTTP 503 with description None


Witness C71500 x [[0.41733333 0.15789474 0.33333333 0.         0.4221408  0.58
  0.48      ]] feasible True M 36.71313770116403 rho 0.9922972728643774 m 0.8517341184014042 ref 0.8517398423396847


Witness C70600 x [[0.214      0.31578947 0.33333333 0.         0.3810528  0.58
  0.51      ]] feasible True M 42.70176296410033 rho 0.9950473086675334 m 0.8079845835716529 ref 0.8077508123376581


In [17]:
# Extend region-split floor search to optionally require a dense candidate.
import inspect
source=inspect.getsource(robust_floor).replace('def robust_floor(', 'def dense_floor(').replace('seed=73):','seed=73,dmin=.95):')
source=source.replace("np.ones(len(z))])","np.ones(len(z)),o['rho']-dmin])")
exec(source)
t0=time.time();extra_results=[]
for label,model,bar,extra,dmin in [
 ('Original CAL, deposited bars',{'p':pbase,'mode':'900','shape':'linear'},'deposited',{},0),
 ('Original CAL, wrought dense bars',{'p':pbase,'mode':'900','shape':'linear'},'wrought',{},0),
 ('700 C refit, require rho >= 0.95',models[15],'deposited',{},.95),
 ('No beta selective penalty',fbase,'deposited',{'isel':0},0),
 ('No porosity multiplier',fbase,'deposited',{'cpor':0},0),
]:
 rr={'label':label}
 for a in refx:rr[a]=dense_floor(model,a,bar,extra,dmin=dmin)
 extra_results.append(rr);print(label,[(a,rr[a]['floor']) for a in refx],flush=True)
(OUT/'floor_bar_and_density_checks.json').write_text(json.dumps(extra_results,default=jsondefault,indent=2))
print('seconds',time.time()-t0)

Original CAL, deposited bars [('C71500', np.float64(12.497307879865335)), ('C70600', np.float64(6.39910818211854))]


Original CAL, wrought dense bars [('C71500', np.float64(15.970076862142438)), ('C70600', np.float64(6.966231899085674))]


700 C refit, require rho >= 0.95 [('C71500', np.float64(16.81987835799603)), ('C70600', np.float64(7.629361520933691))]


No beta selective penalty [('C71500', np.float64(12.506955517837026)), ('C70600', np.float64(2.9869409126166095))]


No porosity multiplier [('C71500', np.float64(15.605911628357594)), ('C70600', np.float64(6.842001407849892))]


seconds 165.45519161224365


In [18]:
for term in ['93.9','84.9','95.1','89.5']:
 j=lt.find(term);print(term,lt[max(0,j-200):j+600])
print('README grid numbers')
for mn in [0,2,4]:print(mn,df.query('Ni==10 and Al==0 and Fe==1.5 and Mn==@mn')[['sch_range99','T_liq','T_sol']].to_dict('records'))
print('Total grid hours',df.wall_s.sum()/3600)
print('no alpha 900 counts',sum(df.alpha_900==0),'NaNs',sum(df.a900_NI.isna()),'liquid 900',sum(df.liquid_900>.001))
print('no-alpha900 phases',df.loc[df.a900_NI.isna(),['Ni','Al','Fe','Mn','alpha_900','beta_900','kappa_nial_900','liquid_900']].head(12).to_string(index=False))
print('Grid point feasibility fraction',TP.feasible(torch.tensor(df[['Ni','Al','Fe','Mn']].values/maxwt)).double().mean().item())
print('sol1 liquid==0',sum(df.sol1_liquid==0),'nuc >0 and sol1 alpha>0.5',sum(((df.sol1_kappa_fe+df.sol1_fcc_feni)>0)&(df.sol1_alpha>.5)))
print('primary label',df.primary.value_counts(dropna=False).to_dict())
for ref in logs:
 print(ref,logs[ref]['landscape'])
# Explicit original witness with exact process settings rounded sufficiently safely.
TP.CAL.update(BASE)
for a,ni,al,P,v in [('C71500',12.51,1.5,.57470,.47967),('C70600',6.41,3,.58280,.51343)]:
 X=candidate([ni,al,1.5,0,P,v]);o=TP.PhysTruth(m_ref=0).full(torch.tensor(X));fast,slow=TP.film_terms(torch.tensor(X[:,:4]))
 print('PASS WITNESS',a,'M',TP.mass_loss_30d(fast,slow,o['beta'],o['rho_m']).item(),'m',o['m'].item(),'bar',TP.reference_m_for(a)[0],'rho',o['rho_m'].item(), 'feasible',TP.feasible(torch.tensor(X)).item())
# demonstrate invalid inputs accepted
bad=torch.tensor([[0,0,0,0,.5,2,-1],[-1,0,0,0,.5,.5,.5],[0,0,0,0,3,.5,.5]])
print('feasible accepts invalid normalized coordinates',TP.feasible(bad).tolist())
# Source lines for generic corrosion constant unit and n=1 vs n=2 issue
j=text.find('Cu + 2Cl');print('Yang copper dissolution valence:',text[j:j+180])

93.9 0 Days Exposure


                     Clean Seawater               Clean Seawater + 2.0 mg/1
Alloy               2.0 m/s Flowrate             Chlorine - 1.5 m/s Flowrate

C70600                      93.9                               48.6
C70600                      84.9                               51.9
C71500                      95.1                               27.7
C71500                      89.5                               31.3
C72200                       16.4                               11.7
C72200                      20.1                               11.7





                                        46
                                  TABLE XVIII

        Comparison of Fouling and Non-Fouling Alloys with Respect to
           Changes in Corrosion Rates with Time During 
84.9 Clean Seawater + 2.0 mg/1
Alloy               2.0 m/s Flowrate             Chlorine - 1.5 m/s Flowrate

C70600                      93.9                               48.6
C70600           

PASS WITNESS C71500 M 36.700350878428296 m 0.8518349762813172 bar 0.8517398423396847 rho 0.9929146180640801 feasible True


PASS WITNESS C70600 M 42.71798675147462 m 0.8078746027387206 bar 0.8077508123376581 rho 0.9950593300947759 feasible True
feasible accepts invalid normalized coordinates [True, True, True]
Yang copper dissolution valence: Cu + 2Cl−→CuCl−2 + e−                                       (2)

the cathode reaction is mainly the process of oxygen reduction

                            O2 + 2H2O + 4e →4OH−   


In [19]:
async def searchcda(a):
 rs=await web_search(f'site.alloys.copper.org/alloy/{a} "Electrical Conductivity" "Thermal Conductivity"',num_results=2)
 print(a,[(r['url'],r['full_text'][-4500:]) for r in rs])
 return a,rs
cda_search=dict(await asyncio.gather(*(searchcda(a) for a in ['C71500','C70600','C61400','C95800'])))

C61400 [('https://alloys.copper.org/alloy/C61400?referrer=facetedsearch', 'gth (ksi) | Torsion Modulus (ksi) | Izod (ft-lbs) | Fatigue Strength** (ksi) | Ultimate Tensile Strength in Shear (ksi) | Section Size (in) |\n| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |\n| * Measured at room temperature, 68°F (20°C). ** Fatigue Strength: 100 x 10 6 cycles, unless indicated as [N] x 10 6. | | | | | | | | | | | | |\n| Flat Products | Hard | H04 | 89 Typ | 60 Typ | 32 Typ | 87 Typ | | | | | | 0.125 |\n| | Hard | H04 | 85 Typ | 58 Typ | 35 Typ | 86 Typ | | | | | | 0.312 |\n| | Hard | H04 | 78 Typ | 45 Typ | 40 Typ | 84 Typ | | | | | | 1 |\n| | Hard | H04 | 80 Typ | 54 Typ | 38 Typ | 85 Typ | | | | | 48 Typ | 0.5 |\n| | Soft Anneal | O60 | 80 Typ | 40 Typ | 40 Typ | | | | | | 42 Typ | 0.315 |\n| | Soft Anneal | O60 | 80 Typ | 40 Typ | 40 Typ | 83 Typ | 42 Typ | | | 26 Typ | | 0.312 |\n| | Soft Anneal | O60 | 82 Typ | 45 Typ | 40 Typ | 84 Typ | 45 Typ | | | 30 Typ 

C70600 [('https://alloys.copper.org/alloy/C70600?referrer=facetedsearch', '5% Ext (ksi) | Elongation (%) | Rockwell B scale | Rockwell F scale | Rockwell 30T scale |\n| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |\n| * Measured at room temperature, 68°F (20°C). | | | | | | | | | |\n| Bar | Hard | H04 | 50 Typ | 25 Typ | | 15 Typ | | | |\n| | Soft Anneal | O60 | 38 Min | | 15 Typ | 30 Typ | | | |\n| Flat Products | 1/4 Hard | H01 | 53 Typ | 16 Typ | | 35 Typ | 60 Typ | | |\n| | As Hot Rolled | M20 | 40 Min for Standard | | 15 Typ | 30 Min for Standard | 3 Min for Standard | | |\n| | Hard | H04 | 75 Typ | 73 Typ | | 5 Typ | 80 Typ | | |\n| Tube | Annealed | O61 | 44 Typ | | 16 Typ | 42 Typ | 15 Typ | 65 Typ | 26 Typ |\n| | Light Drawn, Light Cold Rolled | H55 | 45 Min for Standard | | | 10 Typ | 72 Typ | 100 Typ | 70 Typ |\n\nPhysical Properties\n\n| Melting Point - Liquidus°F | 2100 |\n| --- | --- |\n| Melting Point - Solidus°F | 2010 |\n| Density lb/cu in. at 68°F | 0.32

C71500 [('https://alloys.copper.org/alloy/C71500?referrer=facetedsearch', 'kwell F scale | Section Size (in) | Cold Work (%) |\n| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |\n| * Measured at room temperature, 68°F (20°C). | | | | | | | | | |\n| Flat Products | As Hot Rolled | M20 | 55 Typ | | 45 Typ | 35 Typ | | 1 | |\n| Rod | 1/2 Hard | H02 | 75 Typ | | 15 Typ | 80 Typ | | 1 | 20 |\n| Tube | Nominal Grain Size 0.025 mm | OS025 | 60 Typ | | 45 Typ | 45 Typ | 80 Typ | | |\n| | Nominal Grain Size 0.035 mm | OS035 | 54 Typ | | 45 Typ | 36 Typ | 77 Typ | | |\n\nPhysical Properties\n\n| Melting Point - Liquidus°F | 2260 |\n| --- | --- |\n| Melting Point - Solidus°F | 2140 |\n| Density lb/cu in. at 68°F | 0.323 |\n| Specific Gravity | 8.94 |\n| Electrical Conductivity% IACS at 68°F | 4.6 |\n| Thermal Conductivity Btu/ sq ft/ ft hr/ °F at 68°F | 17 |\n| Coefficient of Thermal Expansion 68-572 10 to -6 power per °F (68 – 572°F) | 9 |\n| Specific Heat Capacity Btu/ lb /°F at 68°

C95800 [('https://alloys.copper.org/alloy/C95800', 'CDA Members-Only Site Maintenance\n\n# This alloys details page is currently under maintenance. Check back soon'), ('https://www.concast.com/files/C95800.pdf', 'C95800\n\nContinuous cast\n\n## Typical uses\n\nNickel-aluminum bronze\n\nProduct description\n\n#### Fasteners\n\nNuts\n\nSolids 1/2" to 9" O.D.\n\n#### Industrial\n\nTubes 1 1/8" to 9" O.D.\n\nRectangles Up to 15"\n\nBushings, gears, machinery, pickling equipment, propeller blades, propeller hub, shafts, valve bodies, wear plates, worm wheels, worms\n\n144"\n\nStandard lengths\n\nShape/form Semi-finished, mill stock or near-net shapes, anode, bar stock, billet/bloom, squares, hex, plate, profile or structural shape, flats/ rectangular bar\n\n#### Marine\n\nCovers for marine hardware, marine hardware, ship building, valves in contact with sea water\n\nSimiliar or equivalent specification\n\n#### Plumbing\n\nCDA ASTM SAE AMS Federal Military Other\n\nElbows\n\n| | B505 | | | |

In [20]:
for a,rs in cda_search.items():
 for r in rs:
  if ('alloys.copper.org' in r['url'] and 'Electrical Conductivity' in r['full_text']) or (a=='C95800' and 'concast.com/files' in r['url']):
   s=r['full_text'];(OUT/f'property_source_{a}.md').write_text(s)
   save_citation('@online{'+('cda' if a!='C95800' else 'concast')+'2026'+a+',\n author={{'+('Copper Development Association' if a!='C95800' else 'Wieland Concast')+'}},\n title={'+r['title'].replace('&','\\&')+'},\n url={'+r['url']+'},\n urldate={'+time.strftime('%Y-%m-%d')+'}\n}')
   print(a,'\n'.join(l for l in s.splitlines() if any(t in l.lower() for t in ['conductivity','solidus','liquidus'])))
wtprops={'C71500':CC.ALLOY['C71500'],'C70600':CC.ALLOY['C70600'],'C61400':CC.ALLOY['C61400'],'C95800':(4.5,9,4.05,1.2)}
for a,w in wtprops.items():
 wt=torch.tensor([w]);av=torch.stack([TP.grid()('a900_'+e,wt) for e in ['NI','AL','FE','MN']],-1)
 at=TP.at_pct(av).numpy()[0];resist=1.7+at@np.array([1.1,.65,2,2.9]);iacs=1.7241/resist*100
 print(a,'a9',av.numpy().round(3),'k',TP.conductivity(av).item(),'IACS',iacs,'k_nominal',TP.conductivity(wt).item())

C71500 | Melting Point - Liquidus°F | 2260 |
| Melting Point - Solidus°F | 2140 |
| Electrical Conductivity% IACS at 68°F | 4.6 |
| Thermal Conductivity Btu/ sq ft/ ft hr/ °F at 68°F | 17 |
| Industrial | Boiler Parts | Conductivity (Thermal) | Thermal Conductivity |
C70600 | Melting Point - Liquidus°F | 2100 |
| Melting Point - Solidus°F | 2010 |
| Electrical Conductivity% IACS at 68°F | 9 |
| Thermal Conductivity Btu/ sq ft/ ft hr/ °F at 68°F | 26 |
| Consumer | Screw Lamp Bases | Conductivity (Electrical) | Electrical Conductivity |
| | Evaporators | Conductivity (Thermal) | Thermal Conductivity |
C61400 | Melting Point - Liquidus°F | 1915 |
| Melting Point - Solidus°F | 1905 |
| Electrical Conductivity% IACS at 68°F | 14 |
| Thermal Conductivity Btu/ sq ft/ ft hr/ °F at 68°F | 39 |
C95800 US customary Metric Melting point – liquidus 1940˚F 1060˚C Melting point – solidus 1910˚F 1043˚C Density 0.276 lb/in3 at 68˚F 7.64 gm/cm3 at 20˚C Specific gravity 7.64 7.64 Electrical conductivity

In [21]:
for idx in [0,13]:
 m=models[idx];ft=np_features(list(alloys.values()),m['mode'],m['shape'])
 for day in [10,30]:
  ar=av_vector(m['p'],0,day*CC.DAY,ft)
  print(m['label'],day,'Yang/C706 ratio',ar[kid['Yang2018']]/ar[kid['C70600']],'rates',ar[kid['Yang2018']]*.0117,ar[kid['C70600']]*.0117)
print('New fit relative changes Yang',models[13]['p']/models[0]['p'])
# Source-reported commercial comparator differs from assumed calibration C70600 Fe and Mn.
ft=np_features([[7,3,1,1],[10,1e-10,1,1]])
for idx in [0,13]:
 m=models[idx]
 for days in [10,30]:
  ar=av_vector(m['p'],0,days*CC.DAY,ft);print('correct comparator',m['label'],days,ar[0]/ar[1])
# schema numeric checks
print('csv duplicates',df[['Ni','Al','Fe','Mn']].duplicated().sum())
print('Tliq<Tsol',sum(df.T_liq<df.T_sol),'sch missing99',df.sch_T99.isna().sum(),'99>=90',sum(df.sch_T99>df.sch_T90))
print('nucleant has primary mismatch',len(df.loc[((df.sol1_kappa_fe+df.sol1_fcc_feni)>0)&~df.primary.fillna('').str.contains('Fe-rich')]))
# Check missing primary maybe eutectic root helper unknown; list extremes.
print('phase sums900 max error',(df.filter(regex='_900$').sum(axis=1)-1).abs().max())
# CDA predicted k relative errors
for a,kref in zip(wtprops,[17*1.730735,26*1.730735,39*1.730735,36]):
 wt=torch.tensor([wtprops[a]]);a9=torch.stack([TP.grid()('a900_'+e,wt) for e in ['NI','AL','FE','MN']],-1);k=TP.conductivity(a9).item()
 print(a,'kref',kref,'prediction',k,'relerror',(k/kref-1)*100)
# exact Faraday conversion
print('one uA day Cu penetration um n2',1e-6*86400*63.546/(2*96485*8.96)*1e4)
# Bound sensitivity fits full table export
rows=[]
for m,r in zip(models,robust_results):
 row={'scenario':m['label'],'weighted_log_SSE':m['loss'],**dict(zip(CC.NAMES,m['p'])),'floor_C71500':r['C71500']['floor'],'floor_C70600':r['C70600']['floor']}
 rows.append(row)
pd.DataFrame(rows).to_csv(OUT/'refit_parameters_and_floors.csv',index=False)
print(pd.DataFrame(rows)[['scenario','floor_C71500','floor_C70600']].to_string(index=False))

Original targets/bounds 10 Yang/C706 ratio 1.13465911096887 rates 0.03025444523419639 0.026663907196199686
Original targets/bounds 30 Yang/C706 ratio 1.2856460691267828 rates 0.020853725474154903 0.016220424870367983
Add Yang 2018 ratio 0.60 at 10 d 10 Yang/C706 ratio 1.0171232044334642 rates 0.02935042895963913 0.028856316355487407
Add Yang 2018 ratio 0.60 at 10 d 30 Yang/C706 ratio 1.1449687153069155 rates 0.020569013562545178 0.017964694831885893
New fit relative changes Yang [3.39226983 1.19765929 1.06484584 0.90702464 1.         1.
 1.08325755]
correct comparator Original targets/bounds 10 1.0249907542764012
correct comparator Original targets/bounds 30 1.0667035818032344
correct comparator Add Yang 2018 ratio 0.60 at 10 d 10 0.9261939071222015
correct comparator Add Yang 2018 ratio 0.60 at 10 d 30 0.9636708572439597
csv duplicates 0
Tliq<Tsol 0 sch missing99 0 99>=90 0
nucleant has primary mismatch 146
phase sums900 max error 1.0000000000065512e-05
C71500 kref 29.422494999999998 

In [22]:
# Add source-matched Yang control composition; expose the unmeasured Fe saturation choice.
alloys['YangControl']=(10,0,1,1); kn=list(alloys);kid={k:i for i,k in enumerate(kn)}
old_np_features=np_features
def np_features(wt,mode='900',ni_shape='linear'):
 if 'cap' not in mode:return old_np_features(wt,mode,ni_shape)
 mat,cap=mode.split('cap');cap=float(cap);wt=np.atleast_2d(wt);f=old_np_features(wt,mat,ni_shape)
 vv=interp(wt);a=np.minimum(wt[:,2],np.maximum(vv[:,FIDX['a'+mat+'_FE']],0))
 f[:,2]=np.minimum(a,cap)/1.5
 return f
# Source-faithful Yang10 comparison, while retaining all original constraints/weights for controlled comparison.
def fit_yang_matched():
 ft=np_features(list(alloys.values()))
 def res(z):
  p=np.exp(z);ar=av_vector(p,0,10*CC.DAY,ft)
  return np.r_[weights*np.log(predictions(p,ft)/target),np.log((ar[kid['Yang2018']]/ar[kid['YangControl']])/(.0215/.0359))]
 rr=least_squares(res,np.log(np.clip(pbase,lo*1.0001,hi/1.0001)),bounds=(np.log(lo),np.log(hi)),max_nfev=800,ftol=1e-11,xtol=1e-11,gtol=1e-9)
 return {'label':'Add Yang 2018, source-matched Fe1 Mn1 control','p':np.exp(rr.x),'loss':sum(rr.fun**2),'mode':'900','shape':'linear','pred':predictions(np.exp(rr.x),ft)}
moremodels=[fit_model('Use actual duplicate-mean 30-d ratio 0.9686',target_=np.r_[178.8/184.6,target[1:]]),fit_yang_matched(),fit_model('Fe protective cap at 3 wt% (not 1.5)',mode='900cap3')]
for m in moremodels:
 rr={'label':m['label'],'loss':m['loss']}
 for a in refx:rr[a]=robust_floor(m,a)
 models.append(m);robust_results.append(rr)
 print(m['label'],'loss',m['loss'],'p',m['p'],'floors',[rr[a]['floor'] for a in refx])
(OUT/'refits.json').write_text(json.dumps(models,default=jsondefault,indent=2));(OUT/'floor_sensitivities_verified.json').write_text(json.dumps(robust_results,default=jsondefault,indent=2))

Use actual duplicate-mean 30-d ratio 0.9686 loss 0.6067672527277386 p [5.00000000e-02 7.23004837e-02 4.89124079e-01 6.42869070e-01
 8.64000000e+05 1.00000000e+05 3.45074469e+00] floors [np.float64(7.605744483942743), np.float64(5.984605708633372)]


Add Yang 2018, source-matched Fe1 Mn1 control loss 0.8462656823067426 p [1.47018510e-01 1.20463328e-01 5.32407838e-01 5.54437036e-01
 8.64000000e+05 1.00000000e+05 3.88382403e+00] floors [np.float64(12.500541527845032), np.float64(4.504937531160897)]


Fe protective cap at 3 wt% (not 1.5) loss 0.5950821361621409 p [5.00000000e-02 1.00327608e-01 5.13020162e-01 5.77273726e-01
 8.64000000e+05 1.00000000e+05 3.60366951e+00] floors [np.float64(10.332817149520466), np.float64(6.1527867095862625)]


25944

In [23]:
# Remaining numerical audit checks and archiving (all originals remain unchanged).
print('NAB liquidus predicted / source',logs['C71500']['references']['C95800 NAB']['T_liq'],1060)
m=models[-2];ft=np_features(list(alloys.values()));ar=av_vector(m['p'],0,10*CC.DAY,ft)
print('Matched Yang fit prediction',ar[kid['Yang2018']]/ar[kid['YangControl']])
wt=rng.random((1000,4))*maxwt
errs={f:np.max(np.abs(TP.grid()(f,torch.tensor(wt)).numpy()-interp(wt)[:,i])) for i,f in enumerate(fields)}
print('Multilinear max error',max(errs.values()))
rows=[]
for m,r in zip(models,robust_results):
 rows.append({'scenario':m['label'],'weighted_log_SSE':m['loss'],**dict(zip(CC.NAMES,m['p'])),'floor_C71500':r['C71500']['floor'],'floor_C70600':r['C70600']['floor'],'rho_C71500':r['C71500'].get('rho'),'rho_C70600':r['C70600'].get('rho')})
pd.DataFrame(rows).to_csv(OUT/'refit_parameters_and_floors.csv',index=False)
header='''from pathlib import Path
import os, sys, types, json, math, time
import numpy as np
import pandas as pd
import torch
from scipy.optimize import least_squares, minimize, differential_evolution
from scipy.interpolate import RegularGridInterpolator
from scipy.special import expit
ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT))
os.chdir(ROOT)
torch.set_num_threads(4)
torch.set_default_dtype(torch.double)
import truth_phys as TP
BASE=TP.CAL.copy()
pkg=types.ModuleType("physics"); pkg.truth_phys=TP; sys.modules["physics"]=pkg
import calibrate_corrosion as CC
OUT=ROOT/"review"
'''
setup='''fields=TP.Grid.FIELDS
V=np.stack([TP.grid().V[k].numpy() for k in fields],axis=-1)
interp=RegularGridInterpolator([TP.AXES[k] for k in ["NI","AL","FE","MN"]],V,bounds_error=False,fill_value=None)
FIDX={k:i for i,k in enumerate(fields)}
maxwt=TP.WT_MAX.numpy()
alloys=CC.ALLOY.copy();alloys["Yang2018"]=(7,3,1,1);alloys["YangControl"]=(10,0,1,1)
kn=list(alloys);kid={k:i for i,k in enumerate(kn)}
target=np.array([1.4,3,2.1,3.1,.7,.36,.18,92.4/11.7,55/11.7,44.2/11.7,29/11.7,37/11.7,2])
weights=np.array([1,1,1,1,.7,.7,.7,.5,.5,.5,.5,.5,.7])
lo=np.array([.05,.05,.05,.05,.25*CC.DAY,1e5,.05]);hi=np.array([1.5]*4+[10*CC.DAY,1e9,10])
pbase=np.array([BASE[k] for k in CC.NAMES])
TT=TP.TGRID.numpy()
bounds=[(0,30),(0,9.5),(0,4.5),(0,4),(0,1),(0,1)]
refx={a:TP.reference_m_for(a)[2].numpy() for a in ["C71500","C70600"]}
'''
fnames=[av_vector,predictions,fit_model,np_outputs,candidate,robust_floor,dense_floor,fit_yang_matched,jsondefault]
parts=[header,setup,inspect.getsource(old_np_features),'old_np_features=np_features\n',inspect.getsource(np_features)]
parts.extend(source if f is dense_floor else inspect.getsource(f) for f in fnames)
parts.append('''
if __name__ == "__main__":
    m=fit_model("Original targets/bounds")
    print("Fitted constants",dict(zip(CC.NAMES,m["p"])))
    for a in refx:
        print(a,"refitted",robust_floor(m,a))
        print(a,"as supplied",robust_floor(dict(p=pbase,mode="900",shape="linear"),a))
        print(a,"wrought bar",robust_floor(dict(p=pbase,mode="900",shape="linear"),a,bar="wrought"))
''')
(OUT/'audit_reproduce.py').write_text('\n\n'.join(parts))
sub=subprocess.run([sys.executable,'-c','import sys;sys.path.insert(0,"/workspace/review");import audit_reproduce as a;print(a.fit_model("verification")["loss"])'],capture_output=True,text=True,timeout=60)
print('Archived script verification',sub.returncode,sub.stdout,sub.stderr[-500:])

NAB liquidus predicted / source 1115.7385333333334 1060
Matched Yang fit prediction 0.9447010886501767
Multilinear max error 6.821210263296962e-13


Archived script verification 0 0.5950821361621409
 


In [24]:
from e14c.web import citation_files
print('Saved BibTeX:',[str(p) for p in citation_files()]);print('audit files:',[(p.name,round(p.stat().st_size/1024,1)) for p in OUT.iterdir() if p.suffix in ('.py','.csv','.json') and p.name in ('audit_reproduce.py','refit_parameters_and_floors.csv','floor_sensitivities_verified.json','floor_bar_and_density_checks.json')]);
# Numerical typo 70-30 C71500 source source values
print('Actual 30-day mean ratio',(93.9+84.9)/(95.1+89.5));print('m gate mass ratio for 0.05',10**(1.5*.05));
# selected changes and row numbers for report
for i in [0,1,2,3,5,7,8,10,12,13,14,15,17,18,19]:
 r=robust_results[i];print(i,models[i]['label'],round(r['C71500']['floor'],2),round(r['C70600']['floor'],2))
print('wrought',[(a,extra_results[1][a]['floor']) for a in refx]);print('No beta',[(a,extra_results[3][a]['floor']) for a in refx])

Saved BibTeX: ['/workspace/citations/local-physics-audit/92e929a9-38ce-45ff-855f-de856bce8435.bib']
audit files: [('audit_reproduce.py', 10.0), ('refit_parameters_and_floors.csv', 5.0), ('floor_sensitivities_verified.json', 25.3), ('floor_bar_and_density_checks.json', 6.2)]
Actual 30-day mean ratio 0.9685807150595884
m gate mass ratio for 0.05 1.1885022274370185
0 Original targets/bounds 12.51 6.4
1 90/70 target 0.95 (actual 30 d) 7.39 5.96
2 90/70 target 2.3 (62 d alternative) 16.73 5.31
3 Drop invented Cu ratio and unidentified 1-y bronze 0.0 0.0
5 2-h measurements full weight; omit 1-y bronze 1.38 0.0
7 Allow tau up to 100 d 15.35 6.53
8 Allow K down to 1 12.69 6.43
10 Fix a_Al = 0.4; refit others 4.98 0.0
12 Fix K = 1e7; refit others 16.03 4.59
13 Add Yang 2018 ratio 0.60 at 10 d 13.17 4.14
14 Ni benefit saturating, scale 10 wt% 5.56 6.96
15 Use 700 C matrix; refit 1.3 0.95
17 Use actual duplicate-mean 30-d ratio 0.9686 7.61 5.98
18 Add Yang 2018, source-matched Fe1 Mn1 control 12.

In [25]:
# Clarify that 0.95 was the supplied approximation, not the recomputed duplicate mean.
for collection in [models,robust_results]:
 for r in collection:
  if r['label']=='90/70 target 0.95 (actual 30 d)':r['label']='90/70 target 0.95 (supplied approximation to 30-d data)'
(OUT/'refits.json').write_text(json.dumps(models,default=jsondefault,indent=2))
(OUT/'floor_sensitivities_verified.json').write_text(json.dumps(robust_results,default=jsondefault,indent=2))
rows=[]
for m,r in zip(models,robust_results):
 rows.append({'scenario':m['label'],'weighted_log_SSE':m['loss'],**dict(zip(CC.NAMES,m['p'])),'floor_C71500':r['C71500']['floor'],'floor_C70600':r['C70600']['floor'],'rho_C71500':r['C71500'].get('rho'),'rho_C70600':r['C70600'].get('rho')})
pd.DataFrame(rows).to_csv(OUT/'refit_parameters_and_floors.csv',index=False)
# Archive instructions and a directly runnable verification loop for every stored refit.
(OUT/'verify_all_floors.py').write_text('''import json
from audit_reproduce import *
models=json.loads((OUT/'refits.json').read_text())
for m in models:
    m['p']=np.array(m['p'])
    print(m['label'],flush=True)
    for alloy in refx:
        print(alloy,robust_floor(m,alloy),flush=True)
''')
(OUT/'REPRODUCE.txt').write_text('''All originals were left unchanged.
Run python review/audit_reproduce.py to refit original targets and optimize both nickel floors, with deposited and wrought-reference bars.
Run python review/verify_all_floors.py to independently rerun all stored-refit floor searches (several minutes).
refits.json contains the alternative fitted constants, original-objective losses where comparable, feature modes, and predicted calibration values.
refit_parameters_and_floors.csv contains numerical results; tau is seconds, compositions are wt%.
floor_sensitivities_verified.json contains numerical floor witnesses, process settings, densities and mass losses.
Run BOFN_REF=C71500 python review/check_phys_adapter.py, and then BOFN_REF=C70600 python review/check_phys_adapter.py, to reproduce available checks. The adapter uses TP.feasible because bofn_alloy was not supplied. The seed-design statistic is explicitly omitted (NaN), not imputed.
Original-run failure logs, reproduced calibration, primary-source texts, and source metadata are also saved here. Citation BibTeX is under /workspace/citations/local-physics-audit/.
Search: a is eliminated analytically by maximizing q; the remaining six variables (four compositions, P, v) use differential evolution plus constrained SLSQP, splitting Al regimes at 1.5 and 7 wt% for discontinuous feasibility rules. These are numerical minima, not formal proofs of global optimality. m saturation is avoided by comparing raw integrated mass loss. References are recomputed under each fit.
''')
print('Final audit artifacts saved to',OUT)

Final audit artifacts saved to /workspace/review
